# Retirement Planner

A self-contained model for projecting your retirement portfolio,
estimating *when* you can retire, and stress-testing that plan against
market variability.

Use the **sidebar** to adjust your assumptions. The deterministic projection
and earliest retirement age update live as you change any input.
Click **Run Monte Carlo** when you want the randomised stress-test results.

**Important modeling choice — everything is in *today's dollars* (real terms).**
Enter your expenses, savings, contributions, and Social Security in today's
purchasing power, and enter *nominal* rates of return together with an inflation
rate — the model converts them to real returns internally via the Fisher equation.
This keeps every dollar amount intuitive: "70k/year" always means 70k of
*today's* buying power, no matter how far in the future.

This is a planning aid, not financial advice. See the **Notes & Caveats**
section at the end for the simplifications baked into this model.

In [ ]:
%matplotlib inline
import json
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Layout, AppLayout
from IPython.display import display

matplotlib.rcParams["figure.dpi"] = 110
pd.options.display.float_format = lambda x: f"{x:,.0f}"

In [ ]:
def project_portfolio(a, retirement_age, return_sequence=None):
    """
    Simulate the portfolio from current_age to life_expectancy.

    Parameters
    ----------
    a : dict
        The assumptions dictionary.
    retirement_age : int
        Age at which contributions stop and withdrawals begin.
    return_sequence : list[float], optional
        One real annual return per simulated year. If omitted, uses the
        constant pre_retirement_return / post_retirement_return from `a`.

    Returns
    -------
    pandas.DataFrame with one row per year of the simulation.
    """
    years = a["life_expectancy"] - a["current_age"]
    balance = a["current_savings"]
    contribution = a["annual_contribution"]
    rows = []

    for i in range(years):
        age = a["current_age"] + i
        retired = age >= retirement_age

        if return_sequence is not None:
            r = return_sequence[i]
        else:
            r = a["post_retirement_return"] if retired else a["pre_retirement_return"]

        ss_income = a["social_security_monthly"] * 12 if age >= a["social_security_start_age"] else 0
        pension_income = (
            a["pension_monthly"] * 12
            if a["pension_start_age"] is not None and age >= a["pension_start_age"]
            else 0
        )

        balance_start = balance

        if retired:
            withdrawal = max(a["annual_expenses"] - ss_income - pension_income, 0)
            balance -= withdrawal
            contrib_this_year = 0
        else:
            withdrawal = 0
            balance += contribution
            contrib_this_year = contribution
            contribution *= (1 + a["contribution_growth_rate"])

        growth = balance * r
        balance += growth

        rows.append({
            "age": age,
            "retired": retired,
            "balance_start": balance_start,
            "contribution": contrib_this_year,
            "ss_income": ss_income,
            "pension_income": pension_income,
            "withdrawal": withdrawal,
            "growth": growth,
            "balance_end": balance,
        })

    return pd.DataFrame(rows)


def find_earliest_retirement_age(a, age_range=None):
    if age_range is None:
        age_range = range(a["current_age"] + 1, a["life_expectancy"])

    for age in age_range:
        df = project_portfolio(a, age)
        if (df["balance_end"] >= 0).all():
            return age, df

    return None, None


def monte_carlo_success(a, retirement_age, n_sims=1000, seed=42, return_paths=False):
    """
    Run n_sims randomized projections for a given retirement age.

    Returns
    -------
    success_rate : float
        Fraction of simulations where the portfolio never goes negative.
    ending_balances : list[float]
        Final balance (at life expectancy) for each simulation.
    paths : np.ndarray, shape (n_sims, years)
        Year-by-year balance for every simulation. Only returned when
        return_paths=True.
    """
    rng = np.random.default_rng(seed)
    years = a["life_expectancy"] - a["current_age"]
    successes = 0
    ending_balances = []
    all_paths = [] if return_paths else None

    for _ in range(n_sims):
        returns = []
        for i in range(years):
            age = a["current_age"] + i
            mean_return = a["post_retirement_return"] if age >= retirement_age else a["pre_retirement_return"]
            returns.append(rng.normal(mean_return, a["return_volatility"]))

        df = project_portfolio(a, retirement_age, return_sequence=returns)
        ending_balances.append(df["balance_end"].iloc[-1])
        if (df["balance_end"] >= 0).all():
            successes += 1
        if return_paths:
            all_paths.append(df["balance_end"].values)

    if return_paths:
        return successes / n_sims, ending_balances, np.array(all_paths)
    return successes / n_sims, ending_balances

def mc_success_grid(a, ages, balances, n_sims=300, seed=42,
                    inflation_rate=0.0, base_age=None):
    """
    Vectorised success-rate grid over retirement ages × starting balances.
    Each cell answers: "if I retire at `age` with `balance`, what fraction
    of simulations survive to life expectancy?"
    When inflation_rate > 0 and base_age is set, `balances` are treated as
    nominal dollars and converted to real for each age column.
    All years use post_retirement_return (person is already retired).
    """
    rng = np.random.default_rng(seed)
    grid = np.zeros((len(balances), len(ages)))

    for j, age in enumerate(ages):
        years = a["life_expectancy"] - age
        if years <= 0:
            grid[:, j] = 1.0
            continue
        ret = rng.normal(a["post_retirement_return"], a["return_volatility"], (n_sims, years))

        # Net withdrawal per year (SS/pension offsets applied)
        withdrawals = np.zeros(years)
        for yr in range(years):
            curr_age = age + yr
            ss = a["social_security_monthly"] * 12 if curr_age >= a["social_security_start_age"] else 0
            pension = (
                a["pension_monthly"] * 12
                if a["pension_start_age"] is not None and curr_age >= a["pension_start_age"]
                else 0
            )
            withdrawals[yr] = max(a["annual_expenses"] - ss - pension, 0)

        if inflation_rate > 0 and base_age is not None:
            real_bals = [b / (1 + inflation_rate) ** (age - base_age) for b in balances]
        else:
            real_bals = balances

        for i, bal in enumerate(real_bals):
            b = np.full(n_sims, float(bal))
            alive = np.ones(n_sims, dtype=bool)
            for yr in range(years):
                b -= withdrawals[yr]
                b *= (1 + ret[:, yr])
                alive &= (b >= 0)
            grid[i, j] = alive.mean()

    return grid

In [ ]:
_style = {"description_width": "170px"}
_layout = Layout(width="280px")

# --- Personal ---
w_current_age = widgets.BoundedIntText(
    value=56, min=18, max=90,
    description="Current age:", style=_style, layout=_layout)
w_target_retirement_age = widgets.BoundedIntText(
    value=65, min=40, max=80,
    description="Target ret. age:", style=_style, layout=_layout)
w_life_expectancy = widgets.BoundedIntText(
    value=90, min=70, max=110,
    description="Life expectancy:", style=_style, layout=_layout)
# --- Savings ---
w_current_savings = widgets.BoundedFloatText(
    value=250, min=0, max=1e6, step=10,
    description="Current savings ($k):", style=_style, layout=_layout)
w_annual_contribution = widgets.BoundedFloatText(
    value=15, min=0, max=500, step=1,
    description="Annual contribution ($k):", style=_style, layout=_layout)
w_contribution_growth_rate = widgets.BoundedFloatText(
    value=2.0, min=0, max=10.0, step=0.5,
    description="Contribution growth (%):", style=_style, layout=_layout)

# --- Returns ---
w_inflation_rate = widgets.BoundedFloatText(
    value=2.5, min=0, max=10.0, step=0.5,
    description="Inflation (%):", style=_style, layout=_layout)
w_pre_retirement_return = widgets.BoundedFloatText(
    value=7.0, min=0, max=20.0, step=0.5,
    description="Pre-ret. nominal (%):", style=_style, layout=_layout)
w_post_retirement_return = widgets.BoundedFloatText(
    value=5.0, min=0, max=15.0, step=0.5,
    description="Post-ret. nominal (%):", style=_style, layout=_layout)
w_return_volatility = widgets.BoundedFloatText(
    value=12.0, min=0, max=30.0, step=0.5,
    description="Return volatility (%):", style=_style, layout=_layout)

# --- Spending / Social Security ---
w_annual_expenses = widgets.BoundedFloatText(
    value=60, min=0, max=1e4, step=1,
    description="Annual expenses ($k):", style=_style, layout=_layout)
w_social_security_monthly = widgets.BoundedFloatText(
    value=1.8, min=0, max=50, step=0.1,
    description="SS monthly ($k):", style=_style, layout=_layout)
w_social_security_start_age = widgets.RadioButtons(
    options=[62, 65, 67, 70], value=67,
    description="SS start age:", style=_style, layout=_layout)

# --- Pension (gated) ---
w_has_pension = widgets.ToggleButton(
    value=False, description="Has pension?",
    button_style="", layout=_layout)
w_pension_monthly = widgets.BoundedFloatText(
    value=0, min=0, max=50, step=0.1,
    description="Pension monthly ($k):", style=_style, layout=_layout)
w_pension_start_age = widgets.BoundedIntText(
    value=60, min=40, max=80,
    description="Pension start age:", style=_style, layout=_layout)
pension_box = widgets.VBox([w_pension_monthly, w_pension_start_age])
pension_box.layout.display = "none"

# --- Monte Carlo controls ---
w_n_sims = widgets.RadioButtons(
    options=[500, 1000, 2000, 5000], value=1000,
    description="# simulations:", style=_style, layout=_layout)
run_mc_button = widgets.Button(
    description="Run Monte Carlo", button_style="primary", layout=_layout)
run_grid_button = widgets.Button(
    description="Run Balance Grid", button_style="info", layout=_layout)

# --- Scenario save/load ---
# Save writes to the notebook's working directory; the full path is shown in
# the status bar after saving so you know exactly where to find the file.
# Load opens an OS file picker — navigate anywhere to pick any .json file.
w_scenario_name = widgets.Text(
    value="scenario", description="Scenario name:",
    style=_style, layout=_layout)
save_scenario_button = widgets.Button(
    description="Save", button_style="", layout=Layout(width="134px"))
load_upload = widgets.FileUpload(
    accept=".json", multiple=False,
    description="Load", layout=Layout(width="134px"))
scenario_status = widgets.HTML(value="")


def _toggle_pension(change=None):
    pension_box.layout.display = "" if w_has_pension.value else "none"


w_has_pension.observe(_toggle_pension, names="value")

def build_assumptions_from_widgets():
    return {
        "current_age": w_current_age.value,
        "life_expectancy": w_life_expectancy.value,
        "current_savings": w_current_savings.value * 1_000,
        "annual_contribution": w_annual_contribution.value * 1_000,
        "contribution_growth_rate": w_contribution_growth_rate.value / 100,
        "pre_retirement_return": (1 + w_pre_retirement_return.value / 100) / (1 + w_inflation_rate.value / 100) - 1,
        "post_retirement_return": (1 + w_post_retirement_return.value / 100) / (1 + w_inflation_rate.value / 100) - 1,
        "return_volatility": w_return_volatility.value / 100,
        "annual_expenses": w_annual_expenses.value * 1_000,
        "social_security_monthly": w_social_security_monthly.value * 1_000,
        "social_security_start_age": w_social_security_start_age.value,
        "pension_monthly": w_pension_monthly.value * 1_000 if w_has_pension.value else 0,
        "pension_start_age": w_pension_start_age.value if w_has_pension.value else None,
        "target_retirement_age": w_target_retirement_age.value,
        "inflation_rate": w_inflation_rate.value / 100,
    }

In [ ]:

# --- Output widgets ---
output_stats          = widgets.Output(layout=Layout(width="100%"))
output_projection_plot = widgets.Output(layout=Layout(width="100%"))
output_earliest_age   = widgets.Output(layout=Layout(width="100%"))
output_mc_stats       = widgets.Output(layout=Layout(width="100%"))
output_mc_fan         = widgets.Output(layout=Layout(width="100%"))
output_sweep_plot     = widgets.Output(layout=Layout(width="100%"))
output_sweep_table    = widgets.Output(layout=Layout(width="100%"))
output_grid_plot      = widgets.Output(layout=Layout(width="100%"))

# --- Sidebar ---
sidebar = widgets.VBox([
    widgets.HTML("<b>Personal</b>"),
    w_current_age,
    w_life_expectancy,
    w_target_retirement_age,
    widgets.HTML("<b>Savings</b>"),
    w_current_savings,
    w_annual_contribution,
    w_contribution_growth_rate,
    widgets.HTML("<b>Returns (nominal)</b>"),
    w_inflation_rate,
    w_pre_retirement_return,
    w_post_retirement_return,
    w_return_volatility,
    widgets.HTML("<b>Spending / Social Security</b>"),
    w_annual_expenses,
    w_social_security_monthly,
    w_social_security_start_age,
    widgets.HTML("<b>Pension</b>"),
    w_has_pension,
    pension_box,
    widgets.HTML("<b>Monte Carlo</b>"),
    w_n_sims,
    run_mc_button,
    run_grid_button,
    widgets.HTML("<b>Scenario</b>"),
    w_scenario_name,
    widgets.HBox([save_scenario_button, load_upload], layout=Layout(width="280px")),
    scenario_status,
], layout=Layout(min_width="300px", max_width="300px", overflow_y="auto", padding="8px"))

# --- Main area ---
main_area = widgets.VBox([
    widgets.HTML("<h3>Deterministic Projection</h3>"),
    output_stats,
    output_projection_plot,
    widgets.HTML("<h3>Earliest Retirement Age</h3>"),
    output_earliest_age,
    widgets.HTML("<h3>Monte Carlo Results</h3>"),
    output_mc_stats,
    output_mc_fan,
    widgets.HTML("<h3>Success Rate vs. Retirement Age</h3>"),
    output_sweep_plot,
    output_sweep_table,
    widgets.HTML("<h3>Success Rate: Age × Balance</h3>"),
    output_grid_plot,
], layout=Layout(flex="1", overflow_y="auto", padding="8px"))

# --- Input Guide area ---
_guide_html = """
<div style="font-family: sans-serif; font-size: 13px; line-height: 1.6; max-width: 820px; padding: 4px 12px;">

<h2 style="margin-top:8px">Input Guide</h2>
<p style="color:#555">All dollar amounts are in <strong>today's dollars</strong> (real terms).
Enter nominal return rates; the model converts them to real returns internally.</p>

<hr/>

<h3 style="color:#2471a3">Personal</h3>

<h4>Current Age</h4>
<p>Your age right now. The model simulates every year from this age to your life expectancy.</p>

<h4>Life Expectancy</h4>
<p>How long your money needs to last. This is the planning horizon, not a prediction.
Using a longer value is conservative — it forces the plan to hold up longer.</p>
<ul>
  <li><strong>Rough guide:</strong> Social Security Administration period life tables show a 65-year-old
      man can expect to live to ~84, a woman to ~87. But half of all people live <em>longer</em>
      than the median.</li>
  <li><strong>Practical choices:</strong> 90 is a common conservative target; 95 if longevity runs in
      your family or you want extra margin.</li>
  <li>Visit <a href="https://www.ssa.gov/oact/population/longevity.html" target="_blank">ssa.gov/oact/population/longevity.html</a>
      for the official life expectancy calculator.</li>
</ul>

<h4>Target Retirement Age</h4>
<p>The age at which you plan to stop working and start drawing down the portfolio.
The <em>Earliest Retirement Age</em> panel tells you the earliest age the deterministic
projection stays solvent — use that as a sanity check on your target.</p>

<hr/>

<h3 style="color:#2471a3">Savings</h3>

<h4>Current Savings ($k)</h4>
<p>Total investable assets today: 401(k), IRA, taxable brokerage, etc.
<em>Do not include</em> home equity, car value, or cash you plan to spend soon.
Enter the number in thousands (e.g. 500 for $500,000).</p>

<h4>Annual Contribution ($k)</h4>
<p>How much you add to the portfolio each year, in today's dollars.
Include employer match if it goes into an investment account.
For 2025, 401(k) employee limit is $23,500; IRA limit is $7,000 ($8,000 if 50+).</p>

<h4>Contribution Growth Rate (%)</h4>
<p>How fast your annual contribution grows each year (in real terms, after inflation).
If you expect raises that keep pace with inflation, set this to 0.
If your savings rate is growing faster than inflation (e.g. paying off a mortgage soon
frees up cash), 1–3% is reasonable.</p>

<hr/>

<h3 style="color:#2471a3">Returns (nominal)</h3>

<p style="background:#eaf4fb; padding:8px; border-left:4px solid #2471a3; border-radius:3px">
<strong>Enter nominal rates here.</strong> The model subtracts inflation using the Fisher equation:
<code>real return ≈ (1 + nominal) / (1 + inflation) − 1</code>.
So a 7% nominal return with 2.5% inflation gives a ~4.4% real return internally.</p>

<h4>Inflation (%)</h4>
<p>Expected long-run inflation rate. The Fed targets 2%; historical U.S. average since 1926
is roughly 3%. A value of 2.5–3% is a reasonable planning assumption.</p>

<h4>Pre-Retirement Nominal Return (%)</h4>
<p>Expected annual return on your portfolio before you retire.
Use the table below as a starting point based on your stock/bond allocation:</p>
<table style="border-collapse:collapse; font-size:12px; margin-bottom:8px">
  <tr style="background:#d6eaf8">
    <th style="padding:4px 10px; text-align:left">Allocation</th>
    <th style="padding:4px 10px; text-align:left">Nominal return range</th>
    <th style="padding:4px 10px; text-align:left">Volatility range</th>
  </tr>
  <tr><td style="padding:4px 10px">100% stocks</td><td style="padding:4px 10px">8–10%</td><td style="padding:4px 10px">15–18%</td></tr>
  <tr style="background:#eaf4fb"><td style="padding:4px 10px">80/20 stocks/bonds</td><td style="padding:4px 10px">7–9%</td><td style="padding:4px 10px">12–15%</td></tr>
  <tr><td style="padding:4px 10px">60/40 stocks/bonds</td><td style="padding:4px 10px">6–7.5%</td><td style="padding:4px 10px">9–12%</td></tr>
  <tr style="background:#eaf4fb"><td style="padding:4px 10px">40/60 stocks/bonds</td><td style="padding:4px 10px">4.5–6%</td><td style="padding:4px 10px">7–10%</td></tr>
  <tr><td style="padding:4px 10px">100% bonds</td><td style="padding:4px 10px">3–5%</td><td style="padding:4px 10px">4–7%</td></tr>
</table>
<p style="font-size:12px;color:#555">Historical U.S. large-cap stocks ≈ 10% nominal; bonds ≈ 5% nominal (Ibbotson data, 1926–2023).
Past performance does not guarantee future results.</p>

<h4>Post-Retirement Nominal Return (%)</h4>
<p>Expected return after you retire. Most people shift to a more conservative allocation
in retirement to reduce sequence-of-returns risk. Set 1–2% lower than your
pre-retirement rate, consistent with a more bond-heavy mix.</p>

<h4>Return Volatility (%)</h4>
<p>Standard deviation of annual returns — a single number applied to both pre- and
post-retirement years. Higher volatility widens the fan chart and lowers the Monte
Carlo success rate for a given average return.
Match this to your pre-retirement allocation from the table above;
the post-retirement shift is captured indirectly by the lower mean return.</p>

<hr/>

<h3 style="color:#2471a3">Spending / Social Security</h3>

<h4>Annual Expenses ($k)</h4>
<p>Total annual spending in retirement, in today's dollars. This is often the hardest
number to estimate. Two common approaches:</p>
<ul>
  <li><strong>Bottom-up:</strong> List expected categories — housing, food, transport, healthcare,
      travel, utilities, insurance — and add them up.</li>
  <li><strong>Rule of thumb:</strong> Many planners use 70–80% of pre-retirement gross income as
      a starting point (lower expenses from commuting, work clothes, mortgage pay-off, etc.).
      Adjust up if you plan significant travel or have high healthcare costs.</li>
</ul>
<p>Don't forget: Medicare Part B premiums (~$185/month in 2025), supplemental insurance,
out-of-pocket costs, and potential long-term care. Healthcare is typically the
largest wildcard in retirement budgets.</p>

<h4>Social Security Monthly ($k)</h4>
<p>Your estimated monthly Social Security benefit at the claiming age you select, in today's dollars.
The most accurate source is your <strong>personal earnings record</strong>:</p>
<ol>
  <li>Create an account at <a href="https://www.ssa.gov/myaccount/" target="_blank">ssa.gov/myaccount</a>.</li>
  <li>View your "Statement" — it shows estimated monthly benefits at 62, FRA, and 70.</li>
</ol>
<p>If you haven't checked recently, the statement projections assume you keep working at
your current earnings until the claiming age. Adjust downward if you plan to retire
early and stop contributing to Social Security.</p>

<h4>Social Security Start Age</h4>
<p>When you plan to start claiming Social Security. The trade-off:</p>
<ul>
  <li><strong>62:</strong> Earliest eligible; benefit reduced ~25–30% vs. FRA permanently.</li>
  <li><strong>67 (Full Retirement Age for those born 1960+):</strong> "Full" benefit as shown on your statement.</li>
  <li><strong>70:</strong> Maximum benefit — increases ~8%/year from FRA to 70 (delayed retirement credits).</li>
</ul>
<p>Claiming later is generally better if you expect to live past your mid-80s and have
assets to bridge the gap. Claiming early makes sense if health is a concern or you
need the income.</p>

<hr/>

<h3 style="color:#2471a3">Pension</h3>

<p>Enable this section if you have a defined-benefit pension. Enter the expected
monthly payment and the age at which it starts. Like expenses and Social Security,
enter the amount in today's dollars (most pension COLAs are partial at best — consult
your plan documents).</p>

<hr/>

<h3 style="color:#2471a3">Monte Carlo</h3>

<h4>Number of Simulations</h4>
<p>How many random market scenarios to run. More simulations give a smoother, more
stable success rate — but take longer to compute.</p>
<ul>
  <li><strong>500:</strong> Fast; good for exploring. Success rates accurate to ±2–3%.</li>
  <li><strong>1000:</strong> Default; good balance of speed and accuracy.</li>
  <li><strong>2000–5000:</strong> Use when you want precise percentile bands or are stress-testing
      a final plan. Each step roughly doubles runtime.</li>
</ul>
<p>The fan chart shows individual paths plus 10/25/75/90th percentile bands.
The 10–90 band captures 80% of outcomes; the 25–75 band (IQR) captures the
middle 50%. Red paths are simulations where the portfolio was depleted.</p>

<hr/>

<h3 style="color:#2471a3">Scenario Save / Load</h3>
<p>Use <strong>Save</strong> to write all current inputs to a JSON file in the notebook's working directory.
The status bar shows the full path after saving. Use <strong>Load</strong> to open any previously saved
scenario file from anywhere on your computer — all widgets update automatically and
the plots refresh.</p>

</div>
"""

help_area = widgets.HTML(
    value=_guide_html,
    layout=Layout(flex="1", overflow_y="auto", padding="8px")
)

# --- Tab: Dashboard | Input Guide ---
content_tabs = widgets.Tab(
    children=[main_area, help_area],
    layout=Layout(flex="1", overflow_y="auto")
)
content_tabs.set_title(0, "Dashboard")
content_tabs.set_title(1, "Input Guide")

app = widgets.HBox(
    [sidebar, content_tabs],
    layout=Layout(width="100%", height="100vh", align_items="flex-start")
)
display(app)


In [ ]:
all_det_widgets = [
    w_current_age, w_life_expectancy, w_current_savings, w_annual_contribution,
    w_contribution_growth_rate, w_inflation_rate, w_pre_retirement_return, w_post_retirement_return,
    w_return_volatility, w_annual_expenses, w_social_security_monthly,
    w_social_security_start_age, w_has_pension, w_pension_monthly, w_pension_start_age,
    w_target_retirement_age,
]


def update_deterministic(change=None):
    a = build_assumptions_from_widgets()
    proj = project_portfolio(a, a["target_retirement_age"])
    depleted = proj.loc[proj["balance_end"] < 0, "age"]
    retire_idx = proj["retired"].idxmax()
    balance_at_retirement = (
        proj.loc[retire_idx - 1, "balance_end"] if retire_idx > 0 else a["current_savings"]
    )

    with output_stats:
        output_stats.clear_output(wait=True)
        print(f"Retirement age tested: {a['target_retirement_age']}")
        print(f"Projected balance at retirement: ${balance_at_retirement/1e3:,.1f}k")
        print(f"Projected balance at life expectancy: ${proj['balance_end'].iloc[-1]/1e3:,.1f}k")
        if not depleted.empty:
            print(f"WARNING: Portfolio runs out at age {int(depleted.iloc[0])}")
        else:
            print("Portfolio lasts through your full life expectancy.")

    with output_projection_plot:
        output_projection_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(proj["age"] + 1, proj["balance_end"], label="Portfolio balance")
        ax.axvline(a["target_retirement_age"], color="gray", linestyle="--",
                   label=f"Retire at {a['target_retirement_age']}")
        ax.axhline(0, color="red", linewidth=0.8)
        ax.set_xlabel("Age")
        ax.set_ylabel("Balance ($k, today's dollars)")
        ax.set_title("Projected Portfolio Balance")
        ax.legend()
        ax.grid(True)
        ax.yaxis.set_major_formatter(lambda x, _: f"${x/1e3:,.0f}k")
        plt.tight_layout()
        plt.show()
        plt.close("all")

    with output_earliest_age:
        output_earliest_age.clear_output(wait=True)
        earliest, earliest_df = find_earliest_retirement_age(a)
        if earliest is not None:
            print(f"Earliest sustainable retirement age (average scenario): {earliest}")
            print(f"Projected balance at life expectancy: ${earliest_df['balance_end'].iloc[-1]/1e3:,.1f}k")
        else:
            print("No retirement age in the tested range fully sustains your spending.")
            print("Consider increasing savings/contributions, lowering expenses, or extending the age range.")


def run_monte_carlo(btn=None):
    run_mc_button.disabled = True
    run_mc_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        rate, balances, paths = monte_carlo_success(
            a, a["target_retirement_age"], n_sims=w_n_sims.value, return_paths=True)

        with output_mc_stats:
            output_mc_stats.clear_output(wait=True)
            print(f"Retirement age tested: {a['target_retirement_age']}")
            print(f"Success rate (portfolio never depleted): {rate:.1%}")
            print(f"Median ending balance:          ${np.median(balances)/1e3:,.1f}k")
            print(f"10th percentile ending balance: ${np.percentile(balances, 10)/1e3:,.1f}k")
            print(f"90th percentile ending balance: ${np.percentile(balances, 90)/1e3:,.1f}k")

        with output_mc_fan:
            output_mc_fan.clear_output(wait=True)
            ages = np.arange(a["current_age"] + 1, a["life_expectancy"] + 1)
            success_mask = (paths >= 0).all(axis=1)
            n_sims_run = len(paths)

            fig, ax = plt.subplots(figsize=(9, 5))

            # Individual paths — faint texture behind the bands
            rng_vis = np.random.default_rng(0)
            n_plot = min(300, n_sims_run)
            plot_idx = rng_vis.choice(n_sims_run, n_plot, replace=False)
            for i in plot_idx:
                clr = "#e74c3c" if not success_mask[i] else "#3498db"
                ax.plot(ages, paths[i] / 1e3, color=clr, alpha=0.04, linewidth=0.5)

            # Percentile bands: outer = 10–90th, inner = 25–75th (IQR)
            pcts = np.percentile(paths, [10, 25, 50, 75, 90], axis=0)

            # Outer band: 10th–90th (light fill + dashed edges)
            ax.fill_between(ages, pcts[0] / 1e3, pcts[4] / 1e3,
                            color="#aed6f1", alpha=0.7, label="10–90th pct")
            ax.plot(ages, pcts[0] / 1e3, color="#5dade2", linewidth=1.0, linestyle="--")
            ax.plot(ages, pcts[4] / 1e3, color="#5dade2", linewidth=1.0, linestyle="--")

            # Inner band: 25th–75th (darker fill + solid edges)
            ax.fill_between(ages, pcts[1] / 1e3, pcts[3] / 1e3,
                            color="#2e86c1", alpha=0.5, label="25–75th pct")
            ax.plot(ages, pcts[1] / 1e3, color="#1a5276", linewidth=1.0, linestyle="-")
            ax.plot(ages, pcts[3] / 1e3, color="#1a5276", linewidth=1.0, linestyle="-")

            # Median
            ax.plot(ages, pcts[2] / 1e3, color="#1a5276", linewidth=2.5, label="Median")

            ax.axvline(a["target_retirement_age"], color="gray", linestyle="--",
                       label=f"Retire at {a['target_retirement_age']}")
            ax.axhline(0, color="red", linewidth=0.8)
            ax.set_xlabel("Age")
            ax.set_ylabel("Balance ($k, today's dollars)")
            n_failed = (~success_mask).sum()
            ax.set_title(
                f"Monte Carlo paths — retire at {a['target_retirement_age']}  "
                f"({rate:.1%} success,  {n_failed}/{n_sims_run} depleted)")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.yaxis.set_major_formatter(lambda x, _: f"${x:,.0f}k")
            plt.tight_layout()
            plt.show()
            plt.close("all")

        # Age sweep — capped at 500 sims/age for speed
        sweep_start = max(a["current_age"] + 1, 50)
        sweep_end = min(a["life_expectancy"], 81)
        sweep_results = []
        for age in range(sweep_start, sweep_end):
            r, _ = monte_carlo_success(a, age, n_sims=500)
            sweep_results.append({"retirement_age": age, "success_rate": r})
        sweep_df = pd.DataFrame(sweep_results)

        with output_sweep_plot:
            output_sweep_plot.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(sweep_df["retirement_age"], sweep_df["success_rate"] * 100, marker="o")
            ax.axhline(90, color="green", linestyle="--", label="90% success")
            ax.axhline(80, color="orange", linestyle="--", label="80% success")
            ax.set_xlabel("Retirement age")
            ax.set_ylabel("Success rate (%)")
            ax.set_title("Plan Success Rate vs. Retirement Age")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()
            plt.close("all")

        with output_sweep_table:
            output_sweep_table.clear_output(wait=True)
            display(sweep_df.style.format({"success_rate": "{:.1%}"}))

    finally:
        run_mc_button.disabled = False
        run_mc_button.description = "Run Monte Carlo"


run_mc_button.on_click(run_monte_carlo)

for w in all_det_widgets:
    w.observe(update_deterministic, names="value")


def run_balance_grid(btn=None):
    run_grid_button.disabled = True
    run_grid_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        ages = list(range(max(a["current_age"] + 1, 50), min(a["life_expectancy"], 81), 2))
        # Y-axis in nominal dollars — user compares to their actual account balance.
        low  = a["current_savings"] * 0.25
        high = a["current_savings"] * 8.0
        balances = list(np.linspace(low, high, 12))

        grid = mc_success_grid(a, ages, balances, n_sims=500,
                               inflation_rate=a["inflation_rate"],
                               base_age=a["current_age"])

        with output_grid_plot:
            output_grid_plot.clear_output(wait=True)
            ages_arr = np.array(ages)
            bals_arr = np.array(balances) / 1e3  # nominal $k
            fig, ax = plt.subplots(figsize=(9, 5))
            cf = ax.contourf(ages_arr, bals_arr, grid * 100,
                             levels=np.linspace(0, 100, 21),
                             cmap="RdYlGn", vmin=0, vmax=100)
            plt.colorbar(cf, ax=ax, label="Success rate (%)")
            cs = ax.contour(ages_arr, bals_arr, grid * 100,
                            levels=[80, 90], colors=["black", "black"], linewidths=2)
            ax.clabel(cs, fmt="%d%%", fontsize=9)
            ax.set_xlabel("Retirement age")
            ax.set_ylabel("Portfolio balance at retirement (nominal \$k)")
            ax.set_title("Monte Carlo success rate: retirement age \u00d7 starting balance (nominal dollars)")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            plt.close("all")
    finally:
        run_grid_button.disabled = False
        run_grid_button.description = "Run Balance Grid"

run_grid_button.on_click(run_balance_grid)


def save_scenario(btn=None):
    data = {
        "current_age": w_current_age.value,
        "life_expectancy": w_life_expectancy.value,
        "target_retirement_age": w_target_retirement_age.value,
        "current_savings": w_current_savings.value,
        "annual_contribution": w_annual_contribution.value,
        "contribution_growth_rate": w_contribution_growth_rate.value,
        "inflation_rate": w_inflation_rate.value,
        "pre_retirement_return": w_pre_retirement_return.value,
        "post_retirement_return": w_post_retirement_return.value,
        "return_volatility": w_return_volatility.value,
        "annual_expenses": w_annual_expenses.value,
        "social_security_monthly": w_social_security_monthly.value,
        "social_security_start_age": w_social_security_start_age.value,
        "has_pension": w_has_pension.value,
        "pension_monthly": w_pension_monthly.value,
        "pension_start_age": w_pension_start_age.value,
        "n_sims": w_n_sims.value,
    }
    name = w_scenario_name.value.strip() or "scenario"
    filename = name if name.endswith(".json") else name + ".json"
    path = os.path.join(os.getcwd(), filename)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    scenario_status.value = f"<span style='color:green'>Saved to {path}</span>"


def _load_scenario(change):
    if not load_upload.value:
        return
    try:
        file_info = load_upload.value[0]   # ipywidgets 8.x API
        data = json.loads(file_info["content"])
        w_scenario_name.value = file_info["name"].replace(".json", "")
        w_current_age.value = data.get("current_age", w_current_age.value)
        w_life_expectancy.value = data.get("life_expectancy", w_life_expectancy.value)
        w_target_retirement_age.value = data.get("target_retirement_age", w_target_retirement_age.value)
        w_current_savings.value = data.get("current_savings", w_current_savings.value)
        w_annual_contribution.value = data.get("annual_contribution", w_annual_contribution.value)
        w_contribution_growth_rate.value = data.get("contribution_growth_rate", w_contribution_growth_rate.value)
        w_inflation_rate.value = data.get("inflation_rate", w_inflation_rate.value)
        w_pre_retirement_return.value = data.get("pre_retirement_return", w_pre_retirement_return.value)
        w_post_retirement_return.value = data.get("post_retirement_return", w_post_retirement_return.value)
        w_return_volatility.value = data.get("return_volatility", w_return_volatility.value)
        w_annual_expenses.value = data.get("annual_expenses", w_annual_expenses.value)
        w_social_security_monthly.value = data.get("social_security_monthly", w_social_security_monthly.value)
        w_social_security_start_age.value = data.get("social_security_start_age", w_social_security_start_age.value)
        w_has_pension.value = data.get("has_pension", w_has_pension.value)
        w_pension_monthly.value = data.get("pension_monthly", w_pension_monthly.value)
        w_pension_start_age.value = data.get("pension_start_age", w_pension_start_age.value)
        w_n_sims.value = data.get("n_sims", w_n_sims.value)
        scenario_status.value = f"<span style='color:green'>Loaded {file_info['name']}</span>"
    except Exception as e:
        scenario_status.value = f"<span style='color:red'>Error: {e}</span>"


save_scenario_button.on_click(save_scenario)
load_upload.observe(_load_scenario, names="value")

update_deterministic()  # initial render on load

## Notes & Caveats

For the full theoretical background — including mathematical derivations,
justification of modeling choices, discussion of limitations, and
references to the academic literature — see
<a href="https://github.com/jlconlin/RetirementPlanner/blob/main/THEORY.md" target="_blank"><strong>THEORY.md</strong></a>.
